# Evaluating event data: real vs simulated vs generated

How close are simulated and generated event data to real event data of the same motion? This notebook loads the results that notebook 10 saves (`results.npz`) and compares them with three families of metrics: temporal (does activity follow the motion), spatial (are the events in the same places, and on the parts that move) and distributional (do the images look like real event images). No diffusion models and no GPU; it runs in seconds, plus a one-time download of the Inception-v3 weights (about 90 MB) for KID.

Set the action in section 0 and run all. If notebook 10 has not been run for that action, the notebook prints `SKIPPED: <path> not found`.

## Setup

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    REPO = "/content/event-perception"
    if not os.path.exists(REPO):
        !git clone https://github.com/allusson/event-perception.git {REPO}
    %cd {REPO}
    !pip install -q -e . torchmetrics torch-fidelity
    sys.path.insert(0, REPO)
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/event-perception-data/human"
else:
    # local: assumes `pip install -e ".[diffusion]"` was run from the repo root (torch, torchmetrics, torch-fidelity for KID)
    os.chdir(os.path.dirname(os.path.abspath("")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
    DATA_DIR = "data/human"

In [ ]:
import importlib.util, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
warnings.filterwarnings("ignore")

from evcam.metrics import chamfer_distance, motion_selectivity, moving_segments, tolerant_prf

METHODS = ("simulated", "generated")
COLORS = {"real": "k", "simulated": "tab:blue", "generated": "tab:orange"}
# limb segments as pairs of the 13 joints of notebook 10 (nose, R/L shoulder, R/L elbow, R/L hip, R/L wrist, R/L knee, R/L ankle)
SEGMENTS = {"right upper arm": (1, 3), "right forearm": (3, 7), "left upper arm": (2, 4), "left forearm": (4, 8),
            "right thigh": (5, 9), "right shin": (9, 11), "left thigh": (6, 10), "left shin": (10, 12)}
T_NOTEBOOK = time.time()

## 0. Configuration

The only cell to edit. Overrides left as `None` are found automatically.

In [ ]:
ACTION = "wave"            # reads <DATA_DIR>/cache/<ACTION>/results.npz, written by notebook 10

# Overrides: leave as None for automatic
EVENT_THRESH = None        # event pixel: value above this in the [0, 1] white-on-black image; None: the value of notebook 10

# Advanced (usually leave as is)
TOL_PX = 3                 # F1 tolerance, canvas px
MOVE_PX = 10               # a limb moves if one of its joints moves more than this between neighbouring frame times, canvas px
LIMB_DIST = 20             # region around a limb segment, canvas px (notebook 10: 0.04 of the canvas side)
PERSON_MARGIN = 0.1        # person region: box of all phone keypoints, grown by this fraction of its size on each side
KID_SUBSETS = 100          # random subsets for the KID mean and standard deviation

### Checklist

Loads the bundle and prints what it holds and which sections will run. All images are on the same 512 px phone canvas (the generation canvas of notebook 10): real events mapped there with the checklist's spatial mapping of notebook 10, simulated events and generated images in it natively. Only frame times with a generated image (a person found by OpenPose) are compared, so every method is evaluated on the same frames.

In [ ]:
for k, v in {"TOL_PX": TOL_PX, "MOVE_PX": MOVE_PX, "LIMB_DIST": LIMB_DIST, "KID_SUBSETS": KID_SUBSETS}.items():
    if not (isinstance(v, (int, float)) and v > 0):
        raise ValueError(f"{k} must be a number > 0; got {v!r}")
if EVENT_THRESH is not None and not 0 < EVENT_THRESH < 1:
    raise ValueError(f"EVENT_THRESH must be None or between 0 and 1, e.g. 0.2; got {EVENT_THRESH!r}")

RESULTS = f"{DATA_DIR}/cache/{ACTION}/results.npz"
skip = None if os.path.exists(RESULTS) else f"{RESULTS} not found (run notebook 10 first)"
kid_skip = next((f"package {p} not installed (pip install torchmetrics torch-fidelity)"
                 for p in ("torch", "torchmetrics", "torch_fidelity") if importlib.util.find_spec(p) is None), None)

if not skip:
    R = dict(np.load(RESULTS))
    thresh = float(R["event_thresh"]) if EVENT_THRESH is None else EVENT_THRESH
    FR = np.flatnonzero(np.isfinite(R["count_gen"]))                  # frame times with a generated image
    times = R["times_s"][FR]
    valid = R["real_valid"]                                           # canvas pixels the DAVIS sees
    if not valid.any():
        skip = "no spatial mapping between the cameras in notebook 10 (real events not on the phone canvas)"
if not skip:
    ev = {"real": R["real_phone_wob"][FR] > thresh, "simulated": (R["sim_phone_wob"][FR] > thresh) & valid,
          "generated": (R["gen_phone"][FR] > thresh) & valid}
    kp = R["kp_phone"][FR]                                            # phone OpenPose joints, canvas px
    size = int(R["canvas_px"])
    xy = kp[np.isfinite(kp).all(-1)]
    x0, y0 = xy.min(0); x1, y1 = xy.max(0)
    mx, my = PERSON_MARGIN * (x1 - x0), PERSON_MARGIN * (y1 - y0)
    person = np.zeros((size, size), bool)
    person[max(int(y0 - my), 0):int(np.ceil(y1 + my)), max(int(x0 - mx), 0):int(np.ceil(x1 + mx))] = True
    person &= valid
    rms_px = float(R["map_rms_canvas_px"])

print("files")
print(f"  {RESULTS:<44} {'found' if os.path.exists(RESULTS) else 'NOT FOUND'}")
if not skip:
    print(f"  {len(R['times_s'])} frame times, {len(FR)} with a generated image (compared), "
          f"{times[0]:.2f} to {times[-1]:.2f} s after the motion start")
    print("values")
    print(f"  canvas                    {size} x {size} px; DAVIS sees {valid.mean():.0%} of it; "
          f"1 DAVIS px = {np.sqrt(abs(np.linalg.det(R['davis_to_canvas'][:2, :2]))):.1f} canvas px")
    print(f"  spatial mapping           {R['map_src']}" + (f" = {rms_px:.1f} canvas px" if np.isfinite(rms_px) else ""))
    print(f"  event pixel               value > {thresh}   {'auto (notebook 10)' if EVENT_THRESH is None else 'manual'}")
    print(f"  F1 tolerance TOL_PX       {TOL_PX} canvas px" + (f"   (the mapping's RMS residual is {rms_px:.1f} px)" if np.isfinite(rms_px) else ""))
    print(f"  person region             box ({x0 - mx:.0f}, {y0 - my:.0f}, {x1 + mx:.0f}, {y1 + my:.0f}) canvas px, "
          f"{person.mean():.0%} of the canvas")
    print(f"  accumulation, threshold   {int(R['acc_ms'])} ms, simulation c = {float(R['c_sim'])}")
print("sections")
for name in ("2 temporal", "3 spatial", "4 motion selectivity", "5 KID", "6 summary"):
    note = "runs"
    if skip:
        note = f"SKIPPED: {skip}"
    elif name.startswith("2") and len(FR) < 3:
        note = "SKIPPED: fewer than 3 frame times"
    elif name.startswith("5") and (kid_skip or len(FR) < 4):
        note = f"SKIPPED: {kid_skip or 'fewer than 4 frame times'}"
    print(f"  {name:<22} {note}")

## 1. What we compare

Per frame time, three event images on the same canvas: **real** (DAVIS events, mapped), **simulated** (ESIM-style, from the phone video) and **generated** (ControlEvents, from the phone skeleton). Real is the reference. An event pixel is a pixel above `EVENT_THRESH` in the white-on-black image of notebook 10 (counts scaled by their 99th percentile; the generated image as is). Simulated and generated images are restricted to the canvas pixels the DAVIS sees.

PSNR is not used: event images are sparse, and their values depend on the accumulation window and the contrast threshold, so a per-pixel error mostly measures the empty background and the scaling, not where the events are.

## 2. Temporal: does activity follow the motion?

Activity per frame time $k$: $c_k$ = events in the accumulation window inside the canvas box (real, simulated) or the sum of the image (generated, which has no events). Each method's series is z-scored, $z_k = (c_k - \bar c) / \sigma_c$, which removes the different units, and compared with real by the Pearson correlation $r = \frac{1}{N} \sum_k z^\text{real}_k z_k$ over the $N$ frame times; $r = 1$: activity rises and falls exactly with the real one.

In [ ]:
if skip:
    print(f"SKIPPED: {skip}")
elif len(FR) < 3:
    print("SKIPPED: fewer than 3 frame times")
else:
    counts = {"real": R["count_real"][FR], "simulated": R["count_sim"][FR], "generated": R["count_gen"][FR]}
    z = {m: (c - c.mean()) / (c.std() + 1e-12) for m, c in counts.items()}
    r_time = {m: float(np.mean(z["real"] * z[m])) for m in METHODS}
    fig, ax = plt.subplots(figsize=(9, 3.2), layout="constrained")
    for m, v in z.items():
        ax.plot(times, v, "o-", ms=3, lw=1, color=COLORS[m], label=m + ("" if m == "real" else f" (r = {r_time[m]:.2f})"))
    ax.set_xlabel("time since the motion start (s)"); ax.set_ylabel("activity (z-scored, no unit)"); ax.legend()
    plt.show()
    print("Pearson r with real: " + ", ".join(f"{m} {r_time[m]:.2f}" for m in METHODS))

Read this as: high $r$ means the method's amount of activity follows the real one over time, regardless of where the events are.

## 3. Spatial: are the events in the same places?

Per frame time, between the event pixels $G$ of a method and $R$ of real, with $d(p, X)$ the distance in canvas px from pixel $p$ to the nearest pixel of $X$ (distance transforms):

- **Symmetric Chamfer distance** (Barrow et al. 1977; Borgefors 1988), canvas px, lower is better: $d_C = \frac{1}{2} \left( \frac{1}{|G|} \sum_{p \in G} d(p, R) + \frac{1}{|R|} \sum_{p \in R} d(p, G) \right)$.
- **Precision, recall, F1 with tolerance** $\tau$ = `TOL_PX`, as for edge maps (Martin, Fowlkes, Malik 2004; here without their one-to-one matching), 0 to 1, higher is better: $P = |\{p \in G : d(p, R) \le \tau\}| / |G|$, $R_\text{ec} = |\{p \in R : d(p, G) \le \tau\}| / |R|$, $F_1 = 2 P R_\text{ec} / (P + R_\text{ec})$.

Real events include sensor noise all over the image and the other methods mostly do not, so the metrics are also computed inside the person region only (box of the phone keypoints of all frame times, grown by `PERSON_MARGIN`). The cameras are mapped by a similarity fitted to OpenPose keypoints, whose RMS residual (checklist) is part of any mismatch; as a reference, the row *real, shifted* compares real with itself shifted diagonally by that residual. Chamfer distance and recall favour images with many event pixels and precision favours few, so read them together (F1).

In [ ]:
def spatial(a_seq, b_seq, region=None):
    # per frame: Chamfer (px), precision, recall, F1 between event pixels a (method) and b (real), optionally inside region
    out = []
    for a, b in zip(a_seq, b_seq):
        if region is not None:
            a, b = a & region, b & region
        out.append((chamfer_distance(a, b),) + tolerant_prf(a, b, TOL_PX))
    return np.array(out)


def shift(img, d):
    # shift a bool image by (d, d) px, rounded to whole pixels
    s = int(round(d))
    out = np.zeros_like(img)
    out[s:, s:] = img[:img.shape[0] - s, :img.shape[1] - s]
    return out


if skip:
    print(f"SKIPPED: {skip}")
else:
    cmp = {m: ev[m] for m in METHODS}
    if np.isfinite(rms_px):
        cmp = {"real, shifted": np.stack([shift(e, rms_px / np.sqrt(2)) & valid for e in ev["real"]]), **cmp}
    sp = {m: spatial(e, ev["real"]) for m, e in cmp.items()}
    sp_person = {m: spatial(e, ev["real"], person) for m, e in cmp.items()}
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.4), layout="constrained")
    for m in cmp:
        c = COLORS.get(m, "0.6")
        ax[0].plot(times, sp[m][:, 0], "o-", ms=3, lw=1, color=c, label=m)
        ax[0].plot(times, sp_person[m][:, 0], "--", lw=1, color=c)
        ax[1].plot(times, sp[m][:, 3], "o-", ms=3, lw=1, color=c, label=m)
        ax[1].plot(times, sp_person[m][:, 3], "--", lw=1, color=c)
    ax[0].set_ylabel("Chamfer distance (canvas px)"); ax[1].set_ylabel(f"F1 at {TOL_PX} px (0 to 1)")
    for a in ax:
        a.set_xlabel("time since the motion start (s)")
    ax[0].legend(title="solid: whole canvas, dashed: person region", fontsize=8, title_fontsize=8)
    plt.show()
    lines = ["| vs real | Chamfer (px) | precision | recall | F1 | Chamfer, person (px) | F1, person |", "|---|---|---|---|---|---|---|"]
    for m in cmp:
        a, b = np.nanmean(sp[m], 0), np.nanmean(sp_person[m], 0)
        lines.append(f"| {m} | {a[0]:.1f} | {a[1]:.2f} | {a[2]:.2f} | {a[3]:.2f} | {b[0]:.1f} | {b[3]:.2f} |")
    display(Markdown(f"Means over {len(FR)} frame times, tolerance {TOL_PX} px"
                     + (f"; mapping residual {rms_px:.1f} px" if np.isfinite(rms_px) else "") + ":\n\n" + "\n".join(lines)))

## 4. Motion selectivity: are the events on the moving body parts?

Per frame time, each limb segment (left and right upper arm, forearm, thigh, shin) is *moving* if one of its joints moved more than `MOVE_PX` between this and a neighbouring frame time, using the phone keypoints, so the same labels apply to all methods on the phone canvas. With $M$ the pixels within `LIMB_DIST` of the moving segments, $S$ those within `LIMB_DIST` of the still segments (and not in $M$) and $E$ the event pixels,

$$\text{selectivity} = \frac{|E \cap M| \, / \, |M|}{|E \cap S| \, / \, |S|}$$

(event density near what moves over event density near what is still; at least one event pixel is counted in $S$, so the ratio stays finite). Above 1: events concentrate on what moves; about 1: events follow the body regardless of motion. Undefined (gap in the curve) when no segment moves or none is still. Ratios are plotted on a log axis and averaged as the geometric mean, $\exp(\overline{\log \text{selectivity}})$. This replaces the arm and leg shares of notebook 10, which depend on the limbs' areas.

Background level: event pixels per pixel outside the person region (within the DAVIS view), $|E \setminus P| \, / \, |V \setminus P|$; real sensor noise shows up here.

In [ ]:
if skip:
    print(f"SKIPPED: {skip}")
else:
    segs = list(SEGMENTS.values())
    split = [moving_segments(kp, k, segs, MOVE_PX) for k in range(len(FR))]
    sel = {m: np.array([motion_selectivity(ev[m][k], kp[k], mv, st, LIMB_DIST) for k, (mv, st) in enumerate(split)])
           for m in ev}
    bg_region = valid & ~person
    background = {m: np.array([e[bg_region].mean() for e in ev[m]]) for m in ev}
    names = {v: k for k, v in SEGMENTS.items()}
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.4), layout="constrained")
    for m in ev:
        ax[0].plot(times, sel[m], "o-", ms=3, lw=1, color=COLORS[m], label=m)
        ax[1].plot(times, background[m], "o-", ms=3, lw=1, color=COLORS[m], label=m)
    ax[0].axhline(1, color="0.7", lw=0.8); ax[0].set_yscale("log")
    ax[0].set_ylabel("selectivity (density ratio, no unit)"); ax[1].set_ylabel("event pixels per pixel (0 to 1)")
    ax[0].set_title("moving vs still limbs"); ax[1].set_title("background, outside the person region")
    for a in ax:
        a.set_xlabel("time since the motion start (s)")
    ax[0].legend(); plt.show()
    n_mv = np.array([len(mv) for mv, _ in split])
    print(f"moving segments per frame time: {n_mv.min()} to {n_mv.max()} of {len(segs)} (median {np.median(n_mv):.0f}); most often moving: "
          + ", ".join(names[s] for s, _ in sorted(((s, sum(s in mv for mv, _ in split)) for s in segs), key=lambda x: -x[1])[:3]))
    gmean = lambda v: float(np.exp(np.nanmean(np.log(v)))) if np.isfinite(v).any() else np.nan
    print("geometric mean selectivity: " + ", ".join(f"{m} {gmean(v):.2f}" for m, v in sel.items()))

## 5. Distribution: do the images look like real event images?

Kernel Inception Distance (KID; Bińkowski et al. 2018): the unbiased squared maximum mean discrepancy between the Inception-v3 (Szegedy et al. 2016) pool features $x$ (real) and $y$ (method) of the two image sets, with the kernel $k(x, y) = (x^\top y / d + 1)^3$, $d = 2048$:

$$\text{KID} = \frac{1}{m(m-1)} \sum_{i \ne j} \left[ k(x_i, x_j) + k(y_i, y_j) \right] - \frac{2}{m^2} \sum_{i, j} k(x_i, y_j),$$

averaged over `KID_SUBSETS` random subsets of $m$ images each (mean $\pm$ standard deviation; 0 = same distribution). It measures appearance, not alignment, so the real images here are on their own DAVIS canvas (no interpolation from the mapping). With about 30 images per method KID is a weak, high-variance signal, unlike the thousands of images behind the FID values (Heusel et al. 2017) in the ControlEvents paper. Computed with `torchmetrics` (`KernelInceptionDistance`, Inception weights from `torch-fidelity`).

In [ ]:
t_sec = time.time()
kid = {}
if skip:
    print(f"SKIPPED: {skip}")
elif kid_skip or len(FR) < 4:
    print(f"SKIPPED: {kid_skip or 'fewer than 4 frame times'}")
else:
    import torch
    from torchmetrics.image.kid import KernelInceptionDistance
    to_u8 = lambda imgs: torch.from_numpy(np.repeat((np.clip(imgs, 0, 1) * 255).astype(np.uint8)[:, None], 3, axis=1))
    real_u8 = to_u8(R["real_davis_wob"][FR])
    m_sub = len(FR) // 2
    for m, imgs in (("simulated", R["sim_phone_wob"][FR]), ("generated", R["gen_phone"][FR])):
        metric = KernelInceptionDistance(subset_size=m_sub, subsets=KID_SUBSETS)
        torch.manual_seed(0)
        metric.update(real_u8, real=True); metric.update(to_u8(imgs), real=False)
        kid[m] = tuple(float(v) for v in metric.compute())
    print(f"KID vs real, {len(FR)} images per set, subsets of {m_sub}: "
          + ", ".join(f"{m} {v[0]:.3f} ± {v[1]:.3f}" for m, v in kid.items()))
print(f"section 5: {time.time() - t_sec:.1f} s")

## 6. Summary

Means over the compared frame times. Temporal $r$ and KID are unitless; Chamfer in canvas px (512 px canvas); F1 from 0 to 1 at `TOL_PX`; selectivity is a ratio of densities (geometric mean); background in event pixels per pixel. *real, shifted*: real vs itself shifted by the mapping's residual, the mismatch the mapping alone can cause.

In [ ]:
if skip:
    print(f"SKIPPED: {skip}")
else:
    f = lambda v, d=2: "n/a" if v is None or not np.isfinite(v) else f"{v:.{d}f}"
    rt = locals().get("r_time", {})
    head = (f"| method | temporal r | Chamfer (px) | F1 at {TOL_PX} px | F1 at {TOL_PX} px, person | selectivity (ratio, geo. mean) "
            "| background (event px / px) | KID ± std |")
    lines = [head, "|---" * 8 + "|"]
    lines.append(f"| real (reference) | 1 | 0 | 1 | 1 | {f(gmean(sel['real']))} | {f(background['real'].mean(), 4)} | 0 |")
    if "real, shifted" in sp:
        a, b = np.nanmean(sp["real, shifted"], 0), np.nanmean(sp_person["real, shifted"], 0)
        lines.append(f"| real, shifted by {rms_px:.1f} px | | {f(a[0], 1)} | {f(a[3])} | {f(b[3])} | | | |")
    for m in METHODS:
        a, b = np.nanmean(sp[m], 0), np.nanmean(sp_person[m], 0)
        k = f"{kid[m][0]:.3f} ± {kid[m][1]:.3f}" if m in kid else "n/a"
        lines.append(f"| {m} | {f(rt.get(m))} | {f(a[0], 1)} | {f(a[3])} | {f(b[3])} | {f(gmean(sel[m]))} "
                     f"| {f(background[m].mean(), 4)} | {k} |")
    display(Markdown(f"**{ACTION}**, {len(FR)} frame times, events over {int(R['acc_ms'])} ms\n\n" + "\n".join(lines)))
print(f"notebook runtime {time.time() - T_NOTEBOOK:.1f} s")

## References

1. Y. Hu, Y. Xue, S. Klenk, D. Cremers, G. Pons-Moll, "ControlEvents: Controllable Synthesis of Event Camera Data with Foundational Prior from Image Diffusion Models", WACV 2026, [arXiv:2509.22864](https://arxiv.org/abs/2509.22864) (FID of generated vs real event images, Table 4).
2. M. Heusel, H. Ramsauer, T. Unterthiner, B. Nessler, S. Hochreiter, "GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium", NeurIPS 2017 (FID).
3. M. Bińkowski, D. J. Sutherland, M. Arbel, A. Gretton, "Demystifying MMD GANs", ICLR 2018 (KID).
4. C. Szegedy, V. Vanhoucke, S. Ioffe, J. Shlens, Z. Wojna, "Rethinking the Inception Architecture for Computer Vision", CVPR 2016, pp. 2818-2826 (Inception-v3).
5. H. G. Barrow, J. M. Tenenbaum, R. C. Bolles, H. C. Wolf, "Parametric Correspondence and Chamfer Matching: Two New Techniques for Image Matching", IJCAI 1977, pp. 659-663.
6. G. Borgefors, "Hierarchical Chamfer Matching: A Parametric Edge Matching Algorithm", IEEE TPAMI 10(6), pp. 849-865, 1988.
7. D. R. Martin, C. C. Fowlkes, J. Malik, "Learning to Detect Natural Image Boundaries Using Local Brightness, Color, and Texture Cues", IEEE TPAMI 26(5), pp. 530-549, 2004 (boundary precision and recall with a distance tolerance).
8. H. Rebecq, D. Gehrig, D. Scaramuzza, "ESIM: an Open Event Camera Simulator", Conference on Robot Learning (CoRL) 2018.
9. Z. Cao, G. Hidalgo, T. Simon, S.-E. Wei, Y. Sheikh, "OpenPose: Realtime Multi-Person 2D Pose Estimation Using Part Affinity Fields", IEEE TPAMI 43(1), 2021 (online 2019).
10. N. B. Detlefsen et al., "TorchMetrics: Measuring Reproducibility in PyTorch", Journal of Open Source Software 7(70), 4101, 2022; A. Obukhov et al., torch-fidelity, 2020, [github.com/toshas/torch-fidelity](https://github.com/toshas/torch-fidelity) (KID implementation and Inception weights).